## Setup

Keep [`example_data.py`](example_data.py), [`data_fidelity_metrics.py`](data_fidelity_metrics.py), and [`missingness_fidelity_metrics.py`](missingness_fidelity_metrics.py) beside this notebook.

In [1]:
#| label: setup
from IPython.display import display

from example_data import make_example_views
from missingness_fidelity_metrics import (
    joint_missingness_details,
    missingness_hellinger_distance,
)

## 1. Metric and inputs

| Property | Value |
|---|---|
| Metric | Joint missingness Hellinger distance |
| Family | Missingness fidelity |
| Origin | Devised in the source study |
| Classifier | None |
| Input | Original $X,Y$ through $M_X,M_Y$ |
| Holdout | None |
| Range | $[0,1]$; smaller is better |

This metric preserves which columns are missing **together** in each row.

## 2. Mathematics

The missingness pattern of row $i$ in $X$ is its entire mask row

$$
m_i^X=((M_X)_{i1},\ldots,(M_X)_{ip})\in\{0,1\}^p
$$

For every pattern $s$ appearing in either table, define

$$
p_s=\frac1{n_X}\sum_{i=1}^{n_X}\mathbf1\{m_i^X=s\},
\qquad
q_s=\frac1{n_Y}\sum_{i=1}^{n_Y}\mathbf1\{m_i^Y=s\}
$$

The joint missingness distance is the Hellinger distance between these aligned pattern distributions:

$$
D_{\mathrm{miss,joint}}(X,Y)=
\sqrt{\frac12\sum_{s\in S}(\sqrt{p_s}-\sqrt{q_s})^2},
$$

where $S$ is the union of patterns observed in $X$ and $Y$.

In [2]:
#| label: joint-missingness

example = make_example_views()
X, Y = example.X, example.Y

details = joint_missingness_details(X, Y)
D_joint = missingness_hellinger_distance(X, Y, mode="joint")

print("Pattern positions follow the column order:", list(X.columns))
display(details.round(4))
print(f"Joint missingness Hellinger distance = {D_joint:.6f}")

Pattern positions follow the column order: ['age', 'income_k', 'region', 'outcome']


,pattern,P_X(pattern),P_Y(pattern)
0,"(0, 0, 0, 0)",0.5000,0.4
1,"(1, 0, 0, 0)",0.1667,0.2
2,"(0, 1, 0, 0)",0.1667,0.2
3,"(0, 0, 1, 0)",0.1667,0.2


Joint missingness Hellinger distance = 0.071161


## 3. Interpretation

The example gives $D_{\mathrm{miss,joint}}(X,Y)\approx0.0712$. Zero would mean the two tables have identical empirical distributions of full missingness patterns. This is stronger than matching marginal missing rates: two tables can have the same rate in every column while missing different combinations of columns in their rows.

## Takeaway

Treat every mask row as one categorical pattern, align the union of patterns, and calculate one Hellinger distance between the two pattern distributions.